# Week 7: K-means Clustering Sweep and Elbow Analysis

This notebook evaluates K-means on the autoencoder embeddings (13 dimensions) using a parameter sweep for $k = 2..20$.

It produces:
- a full inertia table
- a full silhouette table
- an elbow curve
- a silhouette curve
- a final recommendation for the best $k$

The goal is to support the Week 7 clustering report with reproducible evidence.

## Input data

Loads the autoencoder embeddings (13 dimensions) generated in the previous step.

In [1]:
from pathlib import Path

import json

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
if not (project_root / 'data').exists():
    project_root = project_root.parent

ARTIFACTS_DIR = project_root / 'artifacts' / 'week07'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# input_path = ARTIFACTS_DIR / 'week07_autoencoder_embeddings_latent_13.parquet'
input_path = Path("/content/week07_autoencoder_embeddings_latent_13.parquet")
input_path

PosixPath('/content/week07_autoencoder_embeddings_latent_13.parquet')

## Load the autoencoder embeddings

This matrix is the clustering input. We keep `movieId` as an identifier and cluster on the embedding dimensions.

In [2]:
embedding_frame = pl.read_parquet(input_path)
if 'movieId' not in embedding_frame.columns:
    raise ValueError('Expected a movieId column in the autoencoder embeddings file.')

embedding_columns = [column for column in embedding_frame.columns if column != 'movieId']
if len(embedding_columns) < 2:
    raise ValueError('Need at least two embedding dimensions to run clustering.')

X = embedding_frame.select(embedding_columns).to_pandas().astype(float)
movie_ids = embedding_frame.get_column('movieId').to_list()

print(f'Loaded {embedding_frame.height:,} movies with {len(embedding_columns)} autoencoder embedding dimensions from {input_path}')
display(embedding_frame.head(5).to_pandas())

FileNotFoundError: No such file or directory (os error 2): /content/week07_autoencoder_embeddings_latent_13.parquet

## K-means parameter sweep

We sweep $k$ from 2 to 20 and record inertia and silhouette score for each run.

The full table is the main evidence for deciding the best cluster count.

In [ ]:
random_state = 42
k_values = list(range(2, 40))
results = []
labels_by_k = {}

for k in k_values:
    model = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    labels = model.fit_predict(X)
    inertia = float(model.inertia_)
    silhouette = float(silhouette_score(X, labels))
    labels_by_k[k] = labels

    print(f'  k={k}: inertia={inertia:.4f}, silhouette={silhouette:.4f}, cluster_sizes=[{int(pd.Series(labels).value_counts().min())}-{int(pd.Series(labels).value_counts().max())}]')

    results.append({
        'k': k,
        'inertia': inertia,
        'silhouette': silhouette,
        'cluster_size_min': int(pd.Series(labels).value_counts().min()),
        'cluster_size_max': int(pd.Series(labels).value_counts().max()),
    })

metrics_df = pd.DataFrame(results)
metrics_df['inertia_drop'] = metrics_df['inertia'].shift(1) - metrics_df['inertia']
metrics_df['silhouette_delta'] = metrics_df['silhouette'].diff()

metrics_path = ARTIFACTS_DIR / 'week07_kmeans_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

display(metrics_df)
print(f'Saved metrics to {metrics_path}')

  k=2: inertia=3486192.1365, silhouette=0.3347, cluster_sizes=[5640-56783]
  k=3: inertia=3192456.5303, silhouette=0.1735, cluster_sizes=[5609-39112]
  k=4: inertia=2956833.7933, silhouette=0.1783, cluster_sizes=[5566-32747]
  k=5: inertia=2782775.5512, silhouette=0.1732, cluster_sizes=[5538-28506]
  k=6: inertia=2665775.4252, silhouette=0.1128, cluster_sizes=[5308-17711]
  k=7: inertia=2578276.3146, silhouette=0.1109, cluster_sizes=[4696-15676]
  k=8: inertia=2494919.2966, silhouette=0.1181, cluster_sizes=[1322-15602]
  k=9: inertia=2421729.9890, silhouette=0.1174, cluster_sizes=[1251-11722]
  k=10: inertia=2356072.7219, silhouette=0.1248, cluster_sizes=[1223-11992]
  k=11: inertia=2302139.7025, silhouette=0.1256, cluster_sizes=[1394-11049]
  k=12: inertia=2247260.4620, silhouette=0.1300, cluster_sizes=[1173-10505]
  k=13: inertia=2193331.7555, silhouette=0.1310, cluster_sizes=[1322-10141]
  k=14: inertia=2144405.5619, silhouette=0.1326, cluster_sizes=[1118-9837]
  k=15: inertia=21070

,k,inertia,silhouette,cluster_size_min,cluster_size_max,inertia_drop,silhouette_delta
0,2,3.486192e+06,0.334733,5640,56783,NaN,NaN
1,3,3.192457e+06,0.173516,5609,39112,293735.606136,-0.161216
2,4,2.956834e+06,0.178263,5566,32747,235622.737024,0.004747
3,5,2.782776e+06,0.173176,5538,28506,174058.242072,-0.005088
4,6,2.665775e+06,0.112791,5308,17711,117000.125997,-0.060385
5,7,2.578276e+06,0.110936,4696,15676,87499.110613,-0.001855
6,8,2.494919e+06,0.118057,1322,15602,83357.017992,0.007121
7,9,2.421730e+06,0.117446,1251,11722,73189.307588,-0.000611
8,10,2.356073e+06,0.124818,1223,11992,65657.267181,0.007372
9,11,2.302140e+06,0.125579,1394,11049,53933.019318,0.000761


Saved metrics to /artifacts/week07/week07_kmeans_metrics.csv


## Elbow and silhouette plots

The elbow curve shows inertia across $k$, while the silhouette curve shows separation quality. Both should be read together.

In [10]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Elbow curve: inertia vs k', 'Silhouette vs k'))

fig.add_trace(
    go.Scatter(
        x=metrics_df['k'],
        y=metrics_df['inertia'],
        mode='lines+markers',
        name='Inertia',
        line=dict(color='#1f77b4', width=3),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=metrics_df['k'],
        y=metrics_df['silhouette'],
        mode='lines+markers',
        name='Silhouette',
        line=dict(color='#d62728', width=3),
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text='k', dtick=1, row=1, col=1)
fig.update_xaxes(title_text='k', dtick=1, row=1, col=2)
fig.update_yaxes(title_text='Inertia', row=1, col=1)
fig.update_yaxes(title_text='Silhouette score', row=1, col=2)
fig.update_layout(
    title='Week 7 K-means parameter sweep',
    template='plotly_white',
    height=500,
    width=1200,
)

plot_path = ARTIFACTS_DIR / 'week07_kmeans_elbow_and_silhouette.html'
fig.write_html(str(plot_path))
try:
    fig.write_image(str(plot_path.with_suffix('.png')))
except Exception:
    pass

fig.show()
print(f'Saved plots to {plot_path}')

Saved plots to /artifacts/week07/week07_kmeans_elbow_and_silhouette.html


## Decide the best k

Use the table and plots together.

A simple default choice is the k with the highest silhouette score, then check whether that choice also sits near the elbow.

In [11]:
filtered_metrics = metrics_df[metrics_df['k'] != 2]

best_row = filtered_metrics.loc[filtered_metrics['silhouette'].idxmax()]
best_k = int(best_row['k'])
best_k_inertia = float(best_row['inertia'])
best_k_silhouette = float(best_row['silhouette'])

final_model = KMeans(n_clusters=best_k, random_state=random_state, n_init=10)
final_labels = final_model.fit_predict(X)

cluster_sizes = (
    pd.Series(final_labels)
    .value_counts()
    .sort_index()
    .rename_axis('cluster')
    .reset_index(name='count')
)

assignments = pd.DataFrame({
    'movieId': movie_ids,
    'kmeans_cluster': final_labels,
})

assignments_path = ARTIFACTS_DIR / 'week07_kmeans_assignments.csv'
assignments.to_csv(assignments_path, index=False)

print(f'Best k by silhouette (excluding k=2): {best_k}')
print(f'Inertia at best k: {best_k_inertia:.4f}')
print(f'Silhouette at best k: {best_k_silhouette:.4f}')
display(cluster_sizes)
print(f'Saved cluster assignments to {assignments_path}')

Best k by silhouette (excluding k=2): 4
Inertia at best k: 2956833.7933
Silhouette at best k: 0.1783


,cluster,count
0,0,15273
1,1,32747
2,2,8837
3,3,5566


Saved cluster assignments to /artifacts/week07/week07_kmeans_assignments.csv


In [15]:
df_clustered = embedding_frame.clone().to_pandas()
df_clustered["cluster"] = final_labels

In [17]:
numeric_cols = [c for c in df_clustered.columns if c != "movieId"]
numeric_summary = df_clustered.groupby("cluster")[numeric_cols].mean().round(3)
display(numeric_summary)

,ae_1,ae_2,ae_3,ae_4,ae_5,ae_6,ae_7,ae_8,ae_9,ae_10,ae_11,ae_12,ae_13,cluster
cluster,,,,,,,,,,,,,,
0,0.318,-0.508,0.310,0.673,2.352,-0.934,-1.263,-0.946,-0.980,3.329,0.276,-2.337,-1.579,0.0
1,1.557,-0.387,-0.058,-0.047,0.640,-0.737,-0.997,1.041,0.397,1.874,1.319,0.780,-0.412,1.0
2,2.498,-0.014,1.968,2.019,4.941,0.337,0.693,1.965,1.394,2.491,1.064,0.895,0.772,2.0
3,0.669,4.774,7.445,-2.662,2.326,-1.716,-1.793,1.572,2.315,-1.207,5.431,3.800,0.322,3.0


# Cluster Interpretation Analysis

This section maps the K-means cluster assignments back to the original movie metadata to produce interpretable summaries (sizes, numeric feature statistics, genre proportions, distinctive genres, representative movies) and a short semantic label for each cluster.

Key: We analyze the **top-20 one-hot encoded genres** and **raw rating/count data** (already normalized in preprocessing).

In [29]:
# Cluster interpretation using the same one-hot matrix used by the autoencoder notebook
from pathlib import Path
import json as _json
import numpy as np
import pandas as pd
import polars as pl

project_root = Path.cwd()
project_root = Path("/")

# 1) Load Week 5 one-hot feature matrix (same source used by autoencoder training)
feature_candidates = [
    project_root / 'artifacts' / 'week05' / 'week05_pca_feature_matrix.parquet',
    Path('/content/week05_pca_feature_matrix.parquet'),
]
feature_path = next((p for p in feature_candidates if p.exists()), None)
if feature_path is None:
    raise FileNotFoundError('Could not find week05_pca_feature_matrix.parquet in artifacts/week05 or /content')

feature_frame = pl.read_parquet(feature_path).to_pandas()
print(f'Loaded feature frame from {feature_path} with {feature_frame.shape[0]:,} rows')
print(f'Columns: {list(feature_frame.columns)[:20]}...')

# 2) Denormalize rating statistics from Week 5 features
# Week 5 used log1p for count and z-score for mean rating (mean~3.5, std~1.0)
rating_count = np.expm1(feature_frame['rating_count_log']) if 'rating_count_log' in feature_frame.columns else feature_frame.get('rating_count', np.nan)
rating_mean = (feature_frame['avg_rating_z'] * 1.0 + 3.5) if 'avg_rating_z' in feature_frame.columns else feature_frame.get('rating_mean', np.nan)
rating_std = feature_frame['rating_std'] if 'rating_std' in feature_frame.columns else np.nan

processed_stats = pd.DataFrame({
    'movieId': feature_frame['movieId'],
    'rating_mean': rating_mean,
    'rating_count': rating_count,
    'rating_std': rating_std,
})
print(f'Denormalized rating stats from Week 5: {processed_stats.shape[0]:,} rows')

# 3) Load cluster assignments
assign_path = project_root / 'artifacts' / 'week07' / 'week07_kmeans_assignments.csv'
if not assign_path.exists():
    raise FileNotFoundError(f'Cluster assignments not found: {assign_path}')

clustered = pd.read_csv(assign_path)
if 'movieId' not in clustered.columns:
    clustered = clustered.rename(columns={clustered.columns[0]: 'movieId'})
if 'kmeans_cluster' in clustered.columns:
    clustered = clustered.rename(columns={'kmeans_cluster': 'cluster'})
print(f'Loaded cluster assignments for {clustered.shape[0]:,} movies')

# 4) Merge stats + clusters + one-hot genre/tag columns
merged = processed_stats.merge(clustered[['movieId', 'cluster']], on='movieId', how='inner')
prefixed_cols = [c for c in feature_frame.columns if str(c).startswith('genre_') or str(c).startswith('tag_')]
if prefixed_cols:
    merged = merged.merge(feature_frame[['movieId'] + prefixed_cols], on='movieId', how='left')
print(f'Merged data: {merged.shape[0]:,} rows with denormalized stats + genres + tags + clusters')

# 5) Cluster sizes
cluster_counts = merged['cluster'].value_counts().sort_index().rename_axis('cluster').reset_index(name='count')
cluster_counts['percentage'] = (cluster_counts['count'] / cluster_counts['count'].sum() * 100).round(2)
cluster_counts_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_sizes.csv'
cluster_counts.to_csv(cluster_counts_path, index=False)
print('Cluster sizes saved to', cluster_counts_path)

# 6) Numeric summary
numeric_cols = ['rating_mean', 'rating_count', 'rating_std']
numeric_summary = merged.groupby('cluster')[numeric_cols].agg(['mean', 'median', 'std', 'min', 'max']).round(3)
numeric_summary.columns = ['_'.join(col).strip() for col in numeric_summary.columns.values]
numeric_summary_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_numeric_summary.csv'
numeric_summary.reset_index().to_csv(numeric_summary_path, index=False)
print('Numeric summary saved to', numeric_summary_path)

global_means = merged[numeric_cols].mean()
cluster_means = merged.groupby('cluster')[numeric_cols].mean()
diff_from_global = (cluster_means - global_means).round(3)
diff_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_numeric_diff_from_global.csv'
diff_from_global.reset_index().to_csv(diff_path, index=False)
print('Numeric differences from global mean saved to', diff_path)

def _onehot_indicator_columns(df, prefix):
    cols = []
    for c in df.columns:
        if not str(c).startswith(prefix):
            continue
        s = df[c]
        if not pd.api.types.is_numeric_dtype(s):
            continue
        non_null = s.dropna()
        if non_null.empty:
            continue
        # Keep only one-hot/proportion-like columns and exclude logs/counts/spans.
        if ((non_null >= 0) & (non_null <= 1)).all():
            cols.append(c)
    return cols

# 7) Genre/tag proportions from existing one-hot matrix (no parsing)
genre_cols = _onehot_indicator_columns(merged, 'genre_')
tag_cols = _onehot_indicator_columns(merged, 'tag_')
print(f'Found {len(genre_cols)} one-hot/proportion genre columns and {len(tag_cols)} one-hot/proportion tag columns')

if not genre_cols:
    print('Warning: no one-hot genre columns found in week05_pca_feature_matrix.parquet')
if not tag_cols:
    print('Warning: no one-hot tag columns found in week05_pca_feature_matrix.parquet')

if genre_cols:
    genre_prop = merged.groupby('cluster')[genre_cols].mean().round(3)
    genre_prop_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_genre_proportions.csv'
    genre_prop.reset_index().to_csv(genre_prop_path, index=False)
    print('Genre proportions saved to', genre_prop_path)
    top5_genres = {}
    for cl in genre_prop.index:
        top = genre_prop.loc[cl].sort_values(ascending=False).head(5)
        top5_genres[int(cl)] = list(zip(top.index.tolist(), top.values.round(3).tolist()))
else:
    genre_prop = pd.DataFrame()
    top5_genres = {}

if tag_cols:
    tag_prop = merged.groupby('cluster')[tag_cols].mean().round(3)
    tag_prop_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_tag_proportions.csv'
    tag_prop.reset_index().to_csv(tag_prop_path, index=False)
    print('Tag proportions saved to', tag_prop_path)
    top5_tags = {}
    for cl in tag_prop.index:
        top = tag_prop.loc[cl].sort_values(ascending=False).head(5)
        top5_tags[int(cl)] = list(zip(top.index.tolist(), top.values.round(3).tolist()))
else:
    tag_prop = pd.DataFrame()
    top5_tags = {}

# 8) Distinctiveness (cluster proportion minus global proportion)
if genre_cols:
    global_genre = merged[genre_cols].mean()
    distinctive_genre = (genre_prop - global_genre).round(3)
    distinctive_genre_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_genre_distinctiveness.csv'
    distinctive_genre.reset_index().to_csv(distinctive_genre_path, index=False)
    top5_distinctive_genre = {}
    for cl in distinctive_genre.index:
        top = distinctive_genre.loc[cl].sort_values(ascending=False).head(5)
        top5_distinctive_genre[int(cl)] = list(zip(top.index.tolist(), top.values.round(3).tolist()))
else:
    top5_distinctive_genre = {}

if tag_cols:
    global_tag = merged[tag_cols].mean()
    distinctive_tag = (tag_prop - global_tag).round(3)
    distinctive_tag_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_tag_distinctiveness.csv'
    distinctive_tag.reset_index().to_csv(distinctive_tag_path, index=False)
    top5_distinctive_tag = {}
    for cl in distinctive_tag.index:
        top = distinctive_tag.loc[cl].sort_values(ascending=False).head(5)
        top5_distinctive_tag[int(cl)] = list(zip(top.index.tolist(), top.values.round(3).tolist()))
else:
    top5_distinctive_tag = {}

# 9) Representative movies
representative = {}
for cl in sorted(merged['cluster'].unique()):
    sub = merged[merged['cluster'] == cl].copy()
    rand_sample = sub.sample(n=min(10, len(sub)), random_state=42)
    rand_titles = [f"Movie {mid}" for mid in rand_sample['movieId'].tolist()]

    rating_mean_centroid = sub['rating_mean'].mean()
    dists = np.abs(sub['rating_mean'] - rating_mean_centroid)
    idxs = dists.nsmallest(min(10, len(dists)), keep='all').index
    closest = sub.loc[idxs]
    closest_titles = [
        f"Movie {mid} (rating: {rating:.2f})"
        for mid, rating in zip(closest['movieId'].tolist(), closest['rating_mean'].tolist())
    ]

    representative[int(cl)] = {
        'random': rand_titles,
        'closest_by_rating': closest_titles,
    }

# 10) Heuristic semantic labels
cluster_labels = {}
for cl in sorted(merged['cluster'].unique()):
    label_parts = []

    if cl in top5_distinctive_genre and top5_distinctive_genre[cl]:
        label_parts.extend([g.replace('genre_', '').replace('_', ' ').title() for g, _ in top5_distinctive_genre[cl][:2]])
    if cl in top5_distinctive_tag and top5_distinctive_tag[cl]:
        label_parts.extend([t.replace('tag_', '').replace('_', ' ').title() for t, _ in top5_distinctive_tag[cl][:2]])

    cluster_rating = merged.loc[merged['cluster'] == cl, 'rating_mean'].mean()
    global_rating = merged['rating_mean'].mean()
    if cluster_rating > global_rating + 0.1:
        label_parts.append('High-Rated')
    elif cluster_rating < global_rating - 0.1:
        label_parts.append('Lower-Rated')

    label = ' / '.join(label_parts) if label_parts else f'Cluster {cl}'
    description = (
        f"Cluster {cl}: genres={[g for g, _ in top5_distinctive_genre.get(cl, [])[:2]]}; "
        f"tags={[t for t, _ in top5_distinctive_tag.get(cl, [])[:2]]}. "
        f"Mean rating={cluster_rating:.2f}, "
        f"count={merged.loc[merged['cluster'] == cl, 'rating_count'].mean():.0f}."
    )
    cluster_labels[int(cl)] = {'label': label, 'description': description}

# 11) Write markdown report
report_lines = []
report_lines.append('# Cluster Interpretation Report')
report_lines.append('')
report_lines.append('## Data Source Note')
report_lines.append('Rating statistics and one-hot features loaded from Week 5 PCA feature matrix (`week05_pca_feature_matrix.parquet`).')
report_lines.append('')
report_lines.append('## Cluster Overview')
report_lines.append(cluster_counts.to_markdown(index=False))
report_lines.append('')
report_lines.append('## Rating Statistics (denormalized) - Differences from Global Mean')
if not diff_from_global.empty:
    report_lines.append(diff_from_global.to_markdown())
report_lines.append('')
report_lines.append('## Top 5 Genres per Cluster')
for cl, top in top5_genres.items():
    report_lines.append(f'### Cluster {cl} top genres')
    for g, p in top:
        report_lines.append(f'- {g}: {p:.3f}')
    report_lines.append('')

report_lines.append('## Top 5 Tags per Cluster')
for cl, top in top5_tags.items():
    report_lines.append(f'### Cluster {cl} top tags')
    for t, p in top:
        report_lines.append(f'- {t}: {p:.3f}')
    report_lines.append('')

report_lines.append('## Top 5 Distinctive Genres per Cluster')
for cl, top in top5_distinctive_genre.items():
    report_lines.append(f'### Cluster {cl} distinctive genres')
    for g, d in top:
        report_lines.append(f'- {g}: {d:.3f}')
    report_lines.append('')

report_lines.append('## Top 5 Distinctive Tags per Cluster')
for cl, top in top5_distinctive_tag.items():
    report_lines.append(f'### Cluster {cl} distinctive tags')
    for t, d in top:
        report_lines.append(f'- {t}: {d:.3f}')
    report_lines.append('')

report_lines.append('## Representative Movies (random & by rating)')
for cl, vals in representative.items():
    report_lines.append(f'### Cluster {cl} sample movies (random)')
    for t in vals['random']:
        report_lines.append(f'- {t}')
    report_lines.append('')
    report_lines.append(f'### Cluster {cl} sample movies (closest to mean rating)')
    for t in vals['closest_by_rating']:
        report_lines.append(f'- {t}')
    report_lines.append('')

report_lines.append('## Final Cluster Labels (heuristic)')
report_lines.append('| cluster | label | description |')
report_lines.append('|---:|---|---|')
for cl, info in cluster_labels.items():
    report_lines.append(f"| {cl} | {info['label']} | {info['description']} |")

report_path = project_root / 'artifacts' / 'week07' / 'week07_cluster_interpretation.md'
report_path.write_text('\n'.join(report_lines))
print('Wrote cluster interpretation report to', report_path)

# 12) Save representative samples as JSON
with open(project_root / 'artifacts' / 'week07' / 'week07_cluster_representative.json', 'w') as fh:
    _json.dump(representative, fh, indent=2)
print('Wrote representative samples to artifacts/week07/week07_cluster_representative.json')

Loaded feature frame from /content/week05_pca_feature_matrix.parquet with 62,423 rows
Columns: ['movieId', 'title', 'genres_list', 'tag_tokens', 'rating_std', 'release_year_z', 'avg_rating_z', 'rating_count_log', 'tag_event_count_log', 'unique_tag_count_log', 'rating_span_seconds_z', 'tag_span_seconds_z', 'genre_count_log', 'genre_drama', 'genre_comedy', 'genre_thriller', 'genre_romance', 'genre_action', 'genre_horror', 'genre_documentary']...
Denormalized rating stats from Week 5: 62,423 rows
Loaded cluster assignments for 62,423 movies
Merged data: 62,423 rows with denormalized stats + genres + tags + clusters
Cluster sizes saved to /artifacts/week07/week07_cluster_sizes.csv
Numeric summary saved to /artifacts/week07/week07_cluster_numeric_summary.csv
Numeric differences from global mean saved to /artifacts/week07/week07_cluster_numeric_diff_from_global.csv
Found 19 one-hot/proportion genre columns and 20 one-hot/proportion tag columns
Genre proportions saved to /artifacts/week07/wee

In [31]:
# Final check: display Week 7 interpretation artifacts in-notebook
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

project_root = Path.cwd()
project_root = Path("/")
artifacts_dir = project_root / 'artifacts' / 'week07'

files_to_show = {
    'Cluster sizes': artifacts_dir / 'week07_cluster_sizes.csv',
    'Numeric summary': artifacts_dir / 'week07_cluster_numeric_summary.csv',
    'Diff from global': artifacts_dir / 'week07_cluster_numeric_diff_from_global.csv',
    'Genre proportions': artifacts_dir / 'week07_cluster_genre_proportions.csv',
    'Tag proportions': artifacts_dir / 'week07_cluster_tag_proportions.csv',
    'Genre distinctiveness': artifacts_dir / 'week07_cluster_genre_distinctiveness.csv',
    'Tag distinctiveness': artifacts_dir / 'week07_cluster_tag_distinctiveness.csv',
}

for title, path in files_to_show.items():
    print(f'\n=== {title} ===')
    if path.exists():
        df = pd.read_csv(path)
        print(f'{path.name}: {df.shape[0]:,} rows x {df.shape[1]:,} cols')
        display(df.head(10))
    else:
        print(f'Missing: {path}')

report_path = artifacts_dir / 'week07_cluster_interpretation.md'
if report_path.exists():
    print(f'\n=== Interpretation report preview ===')
    report_text = report_path.read_text()
    display(Markdown('\n'.join(report_text.splitlines()[:120])))
else:
    print(f'Missing: {report_path}')


=== Cluster sizes ===
week07_cluster_sizes.csv: 4 rows x 3 cols


,cluster,count,percentage
0,0,15273,24.47
1,1,32747,52.46
2,2,8837,14.16
3,3,5566,8.92



=== Numeric summary ===
week07_cluster_numeric_summary.csv: 4 rows x 16 cols


,cluster,rating_mean_mean,rating_mean_median,rating_mean_std,rating_mean_min,rating_mean_max,rating_count_mean,rating_count_median,rating_count_std,rating_count_min,rating_count_max,rating_std_mean,rating_std_median,rating_std_std,rating_std_min,rating_std_max
0,0,3.230,3.404,1.230,-0.651,6.107,940.822,15.0,3891.020,0.0,81482.0,0.803,0.908,0.468,0.0,3.182
1,1,3.166,3.404,1.465,-0.651,6.107,165.351,3.0,1345.361,0.0,81491.0,0.685,0.764,0.573,0.0,3.182
2,2,3.429,3.606,1.118,-0.651,6.107,548.545,9.0,2654.520,0.0,57309.0,0.887,0.979,0.505,0.0,3.182
3,3,3.796,4.079,1.161,-0.651,6.107,66.236,4.0,766.942,0.0,48377.0,0.720,0.760,0.555,0.0,3.182



=== Diff from global ===
week07_cluster_numeric_diff_from_global.csv: 4 rows x 4 cols


...

,cluster,genre_drama,genre_comedy,genre_thriller,genre_romance,genre_action,genre_horror,genre_documentary,genre_crime,genre_adventure,genre_sci_fi,genre_children,genre_animation,genre_mystery,genre_fantasy,genre_war,genre_western,genre_musical,genre_film_noir,genre_imax
0,0,0.101,-0.138,0.354,-0.037,0.144,0.168,-0.090,0.191,-0.009,0.060,-0.045,-0.044,0.133,-0.010,-0.021,-0.011,-0.016,0.012,0.004
1,1,0.060,0.066,-0.108,0.030,-0.069,-0.043,-0.089,-0.064,-0.050,-0.028,-0.040,-0.046,-0.046,-0.042,0.012,0.006,-0.016,-0.006,-0.003
2,2,-0.166,0.146,-0.127,0.026,0.075,-0.074,-0.086,-0.044,0.238,0.033,0.254,0.272,-0.033,0.198,-0.000,0.013,0.085,0.002,0.003
3,3,-0.364,-0.237,-0.135,-0.120,-0.112,-0.089,0.906,-0.074,-0.053,-0.053,-0.043,-0.040,-0.044,-0.042,-0.010,-0.021,0.003,-0.006,0.004



=== Tag distinctiveness ===
week07_cluster_tag_distinctiveness.csv: 4 rows x 21 cols


,cluster,tag_000_bd_r,tag_001_woman_director,tag_002_murder,tag_003_independent_film,tag_004_comedy,tag_005_nudity_topless,tag_006_based_on_a_book,tag_007_clv,tag_008_drama,...,tag_010_funny,tag_011_violence,tag_012_based_on_novel_or_book,tag_013_revenge,tag_014_musical,tag_015_criterion,tag_016_betamax,tag_017_love,tag_018_family,tag_019_action
0,0,0.077,-0.041,0.099,-0.007,-0.001,0.049,0.034,0.033,-0.005,...,0.000,0.051,0.027,0.037,-0.016,0.010,0.031,0.002,-0.007,0.029
1,1,-0.036,0.015,-0.031,0.011,-0.005,-0.014,-0.014,-0.010,0.011,...,-0.005,-0.016,-0.008,-0.012,-0.015,-0.001,-0.012,0.004,0.001,-0.011
2,2,0.025,-0.028,-0.036,-0.017,0.035,-0.020,0.009,-0.008,-0.020,...,0.028,-0.017,-0.004,-0.009,0.090,-0.011,0.001,-0.007,0.014,-0.003
3,3,-0.034,0.068,-0.029,-0.012,-0.023,-0.023,-0.020,-0.020,-0.018,...,-0.019,-0.016,-0.018,-0.018,-0.012,-0.007,-0.015,-0.017,-0.010,-0.016



=== Interpretation report preview ===


# Cluster Interpretation Report

## Data Source Note
Rating statistics and one-hot features loaded from Week 5 PCA feature matrix (`week05_pca_feature_matrix.parquet`).

## Cluster Overview
|   cluster |   count |   percentage |
|----------:|--------:|-------------:|
|         0 |   15273 |        24.47 |
|         1 |   32747 |        52.46 |
|         2 |    8837 |        14.16 |
|         3 |    5566 |         8.92 |

## Rating Statistics (denormalized) - Differences from Global Mean
|   cluster |   rating_mean |   rating_count |   rating_std |
|----------:|--------------:|---------------:|-------------:|
|         0 |        -0.045 |        540.327 |        0.057 |
|         1 |        -0.109 |       -235.144 |       -0.061 |
|         2 |         0.154 |        148.05  |        0.141 |
|         3 |         0.521 |       -334.258 |       -0.025 |

## Top 5 Genres per Cluster
### Cluster 0 top genres
- genre_drama: 0.511
- genre_thriller: 0.493
- genre_crime: 0.276
- genre_horror: 0.264
- genre_action: 0.262

### Cluster 1 top genres
- genre_drama: 0.470
- genre_comedy: 0.336
- genre_romance: 0.154
- genre_horror: 0.053
- genre_action: 0.049

### Cluster 2 top genres
- genre_comedy: 0.416
- genre_animation: 0.319
- genre_adventure: 0.304
- genre_children: 0.301
- genre_drama: 0.244

### Cluster 3 top genres
- genre_documentary: 0.996
- genre_drama: 0.046
- genre_comedy: 0.033
- genre_musical: 0.020
- genre_war: 0.020

## Top 5 Tags per Cluster
### Cluster 0 top tags
- tag_000_bd_r: 0.140
- tag_002_murder: 0.135
- tag_005_nudity_topless: 0.073
- tag_011_violence: 0.069
- tag_006_based_on_a_book: 0.056

### Cluster 1 top tags
- tag_001_woman_director: 0.071
- tag_003_independent_film: 0.040
- tag_008_drama: 0.032
- tag_000_bd_r: 0.027
- tag_009_romance: 0.023

### Cluster 2 top tags
- tag_014_musical: 0.108
- tag_000_bd_r: 0.088
- tag_004_comedy: 0.061
- tag_010_funny: 0.047
- tag_006_based_on_a_book: 0.031

### Cluster 3 top tags
- tag_001_woman_director: 0.124
- tag_000_bd_r: 0.029
- tag_003_independent_film: 0.017
- tag_015_criterion: 0.011
- tag_002_murder: 0.007

## Top 5 Distinctive Genres per Cluster
### Cluster 0 distinctive genres
- genre_thriller: 0.354
- genre_crime: 0.191
- genre_horror: 0.168
- genre_action: 0.144
- genre_mystery: 0.133

### Cluster 1 distinctive genres
- genre_comedy: 0.066
- genre_drama: 0.060
- genre_romance: 0.030
- genre_war: 0.012
- genre_western: 0.006

### Cluster 2 distinctive genres
- genre_animation: 0.272
- genre_children: 0.254
- genre_adventure: 0.238
- genre_fantasy: 0.198
- genre_comedy: 0.146

### Cluster 3 distinctive genres
- genre_documentary: 0.906
- genre_imax: 0.004
- genre_musical: 0.003
- genre_film_noir: -0.006
- genre_war: -0.010

## Top 5 Distinctive Tags per Cluster
### Cluster 0 distinctive tags
- tag_002_murder: 0.099
- tag_000_bd_r: 0.077
- tag_011_violence: 0.051
- tag_005_nudity_topless: 0.049
- tag_013_revenge: 0.037

### Cluster 1 distinctive tags
- tag_001_woman_director: 0.015
- tag_003_independent_film: 0.011
- tag_008_drama: 0.011

## Week 7 summary

The tables above provide the full parameter sweep evidence for the report.

Use the best silhouette score as the default candidate, then justify the final choice by checking the elbow curve and the cluster-size stability.

In [32]:
!zip -r /artifacts/week07.zip /artifacts/week07

  adding: artifacts/week07/ (stored 0%)
  adding: artifacts/week07/week07_cluster_genre_distinctiveness.csv (deflated 59%)
  adding: artifacts/week07/week07_cluster_sizes.csv (deflated 10%)
  adding: artifacts/week07/week07_kmeans_assignments.csv (deflated 70%)
  adding: artifacts/week07/week07_cluster_interpretation.md (deflated 72%)
  adding: artifacts/week07/week07_kmeans_metrics.csv (deflated 47%)
  adding: artifacts/week07/week07_cluster_representative.json (deflated 83%)
  adding: artifacts/week07/week07_cluster_numeric_diff_from_global.csv (deflated 29%)
  adding: artifacts/week07/week07_cluster_genre_proportions.csv (deflated 60%)
  adding: artifacts/week07/week07_cluster_tag_distinctiveness.csv (deflated 61%)
  adding: artifacts/week07/week07_cluster_tag_proportions.csv (deflated 60%)
  adding: artifacts/week07/week07_kmeans_elbow_and_silhouette.html (deflated 71%)
  adding: artifacts/week07/week07_cluster_numeric_summary.csv (deflated 60%)
